# F2: Qwen2.5-VL-3B с QLoRA rank 16

F2 повторяет F1-fast в одинаковых условиях: те же 2 000 примеров VK, одна эпоха, одинаковый seed, размер изображений и гиперпараметры. Единственное изменение — **LoRA rank 16** вместо 8. Так проверяется, даёт ли более ёмкий адаптер прирост качества на GQA-ru и MMBench-ru.

In [ ]:
import json
import math
import os
from pathlib import Path

import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration, Trainer, TrainingArguments
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training

os.environ['PYTHONIOENCODING'] = 'utf-8'
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DATA = ROOT / 'data' / 'raw' / 'llava_instruct_ru_train.json'
SUBSET_PATH = ROOT / 'data' / 'processed' / 'llava_instruct_ru_first_iteration_seed42.csv'
IMAGE_CACHE = ROOT / 'data' / 'coco_cache' / 'train2017'
MODEL_DIR = ROOT / 'models' / 'f2_qwen_r16_adapter'
CHECKPOINT_DIR = ROOT / 'checkpoints' / 'f2_qwen_r16'
CONTINUED_CHECKPOINT_DIR = ROOT / 'checkpoints' / 'f2_qwen_r16_continued'
RESULTS_DIR = ROOT / 'results'
for directory in (MODEL_DIR, CHECKPOINT_DIR, CONTINUED_CHECKPOINT_DIR, RESULTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'
SEED = 42
TRAINING_EXAMPLES = 2_000
MAX_LENGTH = 1024
assert torch.cuda.is_available(), 'Для обучения нужна CUDA.'
print('GPU:', torch.cuda.get_device_name(0))

## 1. Обучающая выборка

Стратифицированный выбор с `seed=42` повторяет те же 2 000 примеров, что использовались для F1-fast. Все изображения читаются из локального COCO-кэша.

In [ ]:
subset = pd.read_csv(SUBSET_PATH)
raw_records = json.loads(RAW_DATA.read_text(encoding='utf-8'))

def parse_record(record):
    messages = record['conversations']
    question = next(item['value'] for item in messages if item['from'] == 'human')
    answer = next(item['value'] for item in messages if item['from'] == 'gpt')
    return question.replace('<image>\n', '').strip(), answer.strip(), record['image']

rows = []
for row in subset[['row_number', 'type']].itertuples(index=False):
    question, answer, image_path = parse_record(raw_records[int(row.row_number)])
    rows.append({'type': row.type, 'question': question, 'answer': answer, 'image_path': image_path})
full_train_df = pd.DataFrame(rows)
train_df = (full_train_df.groupby('type', group_keys=False)
            .apply(lambda group: group.sample(
                n=round(TRAINING_EXAMPLES * len(group) / len(full_train_df)), random_state=SEED))
            .reset_index(drop=True))
train_df = train_df.sample(n=TRAINING_EXAMPLES, random_state=SEED).reset_index(drop=True)
missing_images = [path for path in train_df.image_path if not (IMAGE_CACHE / Path(path).name).exists()]
assert len(train_df) == TRAINING_EXAMPLES
assert not missing_images, f'В кэше нет изображений: {len(missing_images)}'
display(train_df['type'].value_counts().rename_axis('type').reset_index(name='examples'))

In [ ]:
class VkVlmDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, index):
        return self.frame.iloc[index].to_dict()

def load_local_image(image_path):
    return Image.open(IMAGE_CACHE / Path(image_path).name).convert('RGB')

dataset = VkVlmDataset(train_df)

## 2. QLoRA rank 16

Во F2 меняются только `r=16` и пропорциональный `lora_alpha=32`; остальная конфигурация идентична F1-fast.

In [ ]:
quantization = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16,
)
processor = AutoProcessor.from_pretrained(
    MODEL_ID, min_pixels=256 * 28 * 28, max_pixels=512 * 28 * 28,
)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=quantization, device_map='auto', dtype=torch.float16,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

RECOVERY_CHECKPOINT = CHECKPOINT_DIR / 'checkpoint-50'
RECOVERY_WEIGHTS = RECOVERY_CHECKPOINT / 'adapter_model.safetensors'
RECOVERY_MODE = RECOVERY_WEIGHTS.exists() and not (RECOVERY_CHECKPOINT / 'trainer_state.json').exists()
if RECOVERY_MODE:
    # После аварийного выключения LoRA-веса целы, но optimizer.pt повреждён.
    model = PeftModel.from_pretrained(model, RECOVERY_CHECKPOINT, is_trainable=True)
    print('Загружены сохранённые LoRA-веса F2 с шага 50.')
else:
    model = get_peft_model(model, LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    ))
model.print_trainable_parameters()

## 3. Подготовка обучающих батчей

In [ ]:
class VlmDataCollator:
    def __call__(self, features):
        images = [load_local_image(item['image_path']) for item in features]
        full_texts, prompt_texts = [], []
        for item, image in zip(features, images):
            user = {'role': 'user', 'content': [
                {'type': 'image', 'image': image},
                {'type': 'text', 'text': item['question']},
            ]}
            full = [user, {'role': 'assistant', 'content': [{'type': 'text', 'text': item['answer']}]}]
            full_texts.append(processor.apply_chat_template(full, tokenize=False))
            prompt_texts.append(processor.apply_chat_template(
                [user], tokenize=False, add_generation_prompt=True))
        batch = processor(
            text=full_texts, images=images, padding=True, truncation=True,
            max_length=MAX_LENGTH, return_tensors='pt',
        )
        prompt_batch = processor(
            text=prompt_texts, images=images, padding=True, truncation=True,
            max_length=MAX_LENGTH, return_tensors='pt',
        )
        labels = batch['input_ids'].clone()
        labels[batch['attention_mask'] == 0] = -100
        for i, prompt_length in enumerate(prompt_batch['attention_mask'].sum(dim=1).tolist()):
            labels[i, :prompt_length] = -100
        batch['labels'] = labels
        return batch

collator = VlmDataCollator()

## 4. Обучение F2

Ячейка запускает одну эпоху обучения — около 250 оптимизационных шагов. Если обнаружен аварийный `checkpoint-50`, из него загружаются целые LoRA-веса и выполняются оставшиеся 200 шагов. Новые чекпойнты сохраняются отдельно, чтобы не читать повреждённый optimizer.

In [ ]:
TOTAL_STEPS = math.ceil(len(dataset) / 8)
REMAINING_STEPS = TOTAL_STEPS - 50 if RECOVERY_MODE else -1
OUTPUT_DIR = CONTINUED_CHECKPOINT_DIR if RECOVERY_MODE else CHECKPOINT_DIR

def latest_complete_checkpoint(directory):
    candidates = []
    for path in directory.glob('checkpoint-*'):
        required = ['adapter_model.safetensors', 'optimizer.pt', 'scheduler.pt', 'trainer_state.json']
        if all((path / name).exists() for name in required):
            candidates.append(path)
    return max(candidates, key=lambda path: int(path.name.split('-')[-1]), default=None)

resume_checkpoint = latest_complete_checkpoint(OUTPUT_DIR)
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR), num_train_epochs=1, max_steps=REMAINING_STEPS,
    per_device_train_batch_size=1, gradient_accumulation_steps=8,
    learning_rate=2e-4, lr_scheduler_type='cosine', warmup_steps=8,
    logging_steps=5, save_strategy='steps', save_steps=50, save_total_limit=2,
    fp16=True, gradient_checkpointing=True, optim='paged_adamw_8bit',
    report_to='none', remove_unused_columns=False, seed=SEED,
)
trainer = Trainer(model=model, args=training_args, train_dataset=dataset, data_collator=collator)
if RECOVERY_MODE:
    print(f'Продолжение F2: осталось {REMAINING_STEPS} шагов.')
else:
    print(f'Полный запуск F2: {TOTAL_STEPS} шагов.')
if resume_checkpoint:
    print(f'Возобновление из полноценного чекпойнта: {resume_checkpoint.name}')
trainer.train(resume_from_checkpoint=str(resume_checkpoint) if resume_checkpoint else None)

model.save_pretrained(MODEL_DIR)
processor.save_pretrained(MODEL_DIR)
(RESULTS_DIR / 'f2_training_config.json').write_text(json.dumps({
    'base_model': MODEL_ID, 'examples': len(dataset), 'lora_rank': 16,
    'epochs': 1, 'learning_rate': 2e-4, 'seed': SEED,
    'recovered_from_step': 50 if RECOVERY_MODE else 0,
}, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Адаптер F2 сохранён в: {MODEL_DIR}')